In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import glob
import json
import random

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, classification_report

설정

In [3]:
DATA_DIR = "/content/drive/MyDrive/salpim_project/04_skeleton_dataset_hip_body_norm"

BATCH_SIZE = 32
EPOCHS = 30
LR = 0.001

# ST-GCN 설정
NUM_JOINTS = 17
IN_CHANNELS = 6
DROPOUT = 0.5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


## (5/20 수)
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


device: cuda


Dataset 만들기

In [4]:
from torch.utils.data import Dataset
import numpy as np
import torch

class SkeletonNPYDataset(Dataset):

    def __init__(self, x_path, y_path):

        self.data = np.load(x_path).astype(np.float32)
        self.labels = np.load(y_path).astype(np.int64)


        self.data = np.transpose(
            self.data,
            (0, 3, 1, 2)
        )

        self.data = torch.tensor(
            self.data,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            self.labels,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

DataLoader 만들기

In [5]:
###(5/20 수정) DataLoader seed 고정

train_dataset = SkeletonNPYDataset(
    os.path.join(DATA_DIR, "train", "X_train.npy"),
    os.path.join(DATA_DIR, "train", "y_train.npy")
)

val_dataset = SkeletonNPYDataset(
    os.path.join(DATA_DIR, "val", "X_val.npy"),
    os.path.join(DATA_DIR, "val", "y_val.npy")
)

test_dataset = SkeletonNPYDataset(
    os.path.join(DATA_DIR, "test", "X_test.npy"),
    os.path.join(DATA_DIR, "test", "y_test.npy")
)


g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=g
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("train 개수:", len(train_dataset))
print("val 개수:", len(val_dataset))
print("test 개수:", len(test_dataset))


train 개수: 629
val 개수: 232
test 개수: 171


입력 크기 확인

In [6]:
sample_x, sample_y = train_dataset[0]

print("sample_x shape:", sample_x.shape)
print("sample_y:", sample_y)

# ST-GCN 입력 형태: (C, T, V)
IN_CHANNELS = sample_x.shape[0]
SEQ_LEN = sample_x.shape[1]
NUM_JOINTS = sample_x.shape[2]
NUM_CLASSES = 10

print("IN_CHANNELS:", IN_CHANNELS)
print("SEQ_LEN:", SEQ_LEN)
print("NUM_JOINTS:", NUM_JOINTS)
print("NUM_CLASSES:", NUM_CLASSES)


sample_x shape: torch.Size([6, 90, 17])
sample_y: tensor(0)
IN_CHANNELS: 6
SEQ_LEN: 90
NUM_JOINTS: 17
NUM_CLASSES: 10


ST-GCN 모델

In [7]:
def build_coco17_adjacency(num_joints=17):
    """
    COCO 17 keypoint 기준 관절 연결 그래프 생성
    0 nose
    1 left_eye, 2 right_eye
    3 left_ear, 4 right_ear
    5 left_shoulder, 6 right_shoulder
    7 left_elbow, 8 right_elbow
    9 left_wrist, 10 right_wrist
    11 left_hip, 12 right_hip
    13 left_knee, 14 right_knee
    15 left_ankle, 16 right_ankle
    """

    edges = [
        (0, 1), (0, 2),
        (1, 3), (2, 4),
        (5, 6),
        (5, 7), (7, 9),
        (6, 8), (8, 10),
        (5, 11), (6, 12),
        (11, 12),
        (11, 13), (13, 15),
        (12, 14), (14, 16),
    ]

    A = np.eye(num_joints, dtype=np.float32)

    for i, j in edges:
        A[i, j] = 1
        A[j, i] = 1

    # degree normalization: D^(-1/2) A D^(-1/2)
    D = np.sum(A, axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(D + 1e-6))
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt

    return torch.tensor(A_norm, dtype=torch.float32)


class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, A, stride=1, dropout=0.5):
        super().__init__()

        self.register_buffer("A", A)

        # spatial graph convolution
        self.gcn = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1
        )

        # temporal convolution
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=(9, 1),
                stride=(stride, 1),
                padding=(4, 0)
            ),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout)
        )

        if in_channels == out_channels and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=(stride, 1)
                ),
                nn.BatchNorm2d(out_channels)
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # x: (N, C, T, V)

        # graph convolution
        # 각 관절 feature를 인접 관절 feature와 섞음
        x_g = torch.einsum("nctv,vw->nctw", x, self.A)
        x_g = self.gcn(x_g)

        # temporal convolution + residual
        out = self.tcn(x_g) + self.residual(x)

        return self.relu(out)


class STGCNActionClassifier(nn.Module):
    def __init__(self, in_channels, num_classes, num_joints=17, dropout=0.5):
        super().__init__()

        A = build_coco17_adjacency(num_joints)

        self.data_bn = nn.BatchNorm1d(in_channels * num_joints)

        self.layer1 = STGCNBlock(in_channels, 64, A, stride=1, dropout=dropout)
        self.layer2 = STGCNBlock(64, 64, A, stride=1, dropout=dropout)
        self.layer3 = STGCNBlock(64, 128, A, stride=2, dropout=dropout)
        self.layer4 = STGCNBlock(128, 128, A, stride=1, dropout=dropout)
        self.layer5 = STGCNBlock(128, 256, A, stride=2, dropout=dropout)

        #self.fc = nn.Identity() ###### 수정
        self.fc = nn.Linear(
            256,
            num_classes
        )

    def forward(self, x):
        # x: (N, C, T, V)
        N, C, T, V = x.shape

        # BatchNorm을 위해 (N, C, T, V) -> (N, C*V, T)
        x = x.permute(0, 1, 3, 2).contiguous()
        x = x.view(N, C * V, T)
        x = self.data_bn(x)

        # 다시 (N, C, T, V)
        x = x.view(N, C, V, T)
        x = x.permute(0, 1, 3, 2).contiguous()

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)

        # global average pooling over time and joints
        x = x.mean(dim=[2, 3])

        logits = self.fc(x)

        return logits


모델 생성

In [8]:
model = STGCNActionClassifier(
    in_channels=IN_CHANNELS,
    num_classes=NUM_CLASSES,
    num_joints=NUM_JOINTS,
    dropout=DROPOUT
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(model)


STGCNActionClassifier(
  (data_bn): BatchNorm1d(102, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1): STGCNBlock(
    (gcn): Conv2d(6, 64, kernel_size=(1, 1), stride=(1, 1))
    (tcn): Sequential(
      (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(9, 1), stride=(1, 1), padding=(4, 0))
      (3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (4): Dropout(p=0.5, inplace=False)
    )
    (residual): Sequential(
      (0): Conv2d(6, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (relu): ReLU(inplace=True)
  )
  (layer2): STGCNBlock(
    (gcn): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (tcn): Sequential(
      (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (1): ReLU(inpla

학습 함수

In [9]:
def train_one_epoch(model, loader):
    model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)

    return avg_loss, acc

평가 함수

In [10]:
def evaluate(model, loader):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            outputs = model(x)
            loss = criterion(outputs, y)

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)

    return avg_loss, acc

학습 실행

In [11]:
import os
import json
import torch

SAVE_DIR = "/content/drive/MyDrive/salpim_project/02_models"
os.makedirs(SAVE_DIR, exist_ok=True)

In [12]:
best_val_acc = 0

# Early Stopping 설정
patience = 5
counter = 0

MODEL_PATH = "/content/drive/MyDrive/salpim_project/02_models/stgcn_mediapipe_pose_baseline.pth"

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
    )

    # validation accuracy 개선 시
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            MODEL_PATH
        )

        print("best ST-GCN model saved")

        # 성능 개선되었으므로 counter 초기화
        counter = 0

    else:
        # 성능 개선 안되면 증가
        counter += 1

        print(f"EarlyStopping Counter: {counter}/{patience}")

    # patience 초과 시 학습 종료
    if counter >= patience:
        print("Early Stopping")
        break


Epoch [1/30] Train Loss: 1.7735, Train Acc: 0.4054 | Val Loss: 3.2204, Val Acc: 0.2026
best ST-GCN model saved
Epoch [2/30] Train Loss: 1.2419, Train Acc: 0.5692 | Val Loss: 1.5935, Val Acc: 0.4957
best ST-GCN model saved
Epoch [3/30] Train Loss: 1.1135, Train Acc: 0.6137 | Val Loss: 1.9287, Val Acc: 0.4483
EarlyStopping Counter: 1/5
Epoch [4/30] Train Loss: 1.1244, Train Acc: 0.6025 | Val Loss: 1.4932, Val Acc: 0.5216
best ST-GCN model saved
Epoch [5/30] Train Loss: 0.9877, Train Acc: 0.6280 | Val Loss: 1.3118, Val Acc: 0.5603
best ST-GCN model saved
Epoch [6/30] Train Loss: 0.9920, Train Acc: 0.6598 | Val Loss: 1.4186, Val Acc: 0.5560
EarlyStopping Counter: 1/5
Epoch [7/30] Train Loss: 0.9062, Train Acc: 0.6582 | Val Loss: 1.4717, Val Acc: 0.5043
EarlyStopping Counter: 2/5
Epoch [8/30] Train Loss: 0.9303, Train Acc: 0.6804 | Val Loss: 1.1432, Val Acc: 0.6509
best ST-GCN model saved
Epoch [9/30] Train Loss: 0.8639, Train Acc: 0.6836 | Val Loss: 1.1578, Val Acc: 0.6250
EarlyStopping Co

라벨 저장

In [13]:
import json
import os

with open(
    os.path.join(DATA_DIR, "label_map.json"),
    "r",
    encoding="utf-8"
) as f:
    label_map = json.load(f)

idx_to_label = {
    int(v): k
    for k, v in label_map.items()
}

with open(
    "/content/drive/MyDrive/salpim_project/02_models/label_map_mediapipe_pose_baseline.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(idx_to_label, f, ensure_ascii=False, indent=4)

print(idx_to_label)

{0: 'A010', 1: 'A011', 2: 'A016', 3: 'A018', 4: 'A023', 5: 'A031', 6: 'A035', 7: 'A041', 8: 'A053', 9: 'A054'}


최종 테스트

In [14]:
MODEL_PATH = "/content/drive/MyDrive/salpim_project/02_models/stgcn_mediapipe_pose_baseline.pth"

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)

test_loss, test_acc = evaluate(model, test_loader)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)


Test Loss: 1.1439794103304546
Test Accuracy: 0.672514619883041


클래스별 성능 확인

In [15]:
import json
import os
from sklearn.metrics import classification_report

with open(
    os.path.join(DATA_DIR, "label_map.json"),
    "r",
    encoding="utf-8"
) as f:
    label_map = json.load(f)

idx_to_label = {
    int(v): k
    for k, v in label_map.items()
}

target_names = [
    idx_to_label[i]
    for i in range(len(idx_to_label))
]

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:

        x = x.to(DEVICE)

        outputs = model(x)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())

label_names = [
    "A010_이빨닦기",
    "A011_손씻기",
    "A016_머리빗기",
    "A018_상의입기",
    "A023_진공청소기",
    "A031_TV리모컨",
    "A035_전화통화",
    "A041_맨손체조",
    "A053_쓰러지기",
    "A054_누웠다일어나기"
]

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=label_names,
        digits=4
    )
)

              precision    recall  f1-score   support

   A010_이빨닦기     0.6154    0.5000    0.5517        16
    A011_손씻기     0.6111    0.7857    0.6875        14
   A016_머리빗기     0.6875    0.5789    0.6286        19
   A018_상의입기     0.7619    0.7273    0.7442        22
  A023_진공청소기     0.8421    0.8889    0.8649        18
  A031_TV리모컨     0.6842    0.6190    0.6500        21
   A035_전화통화     0.1818    0.1429    0.1600        14
   A041_맨손체조     0.5833    0.6364    0.6087        11
   A053_쓰러지기     0.7727    0.8947    0.8293        19
A054_누웠다일어나기     0.7000    0.8235    0.7568        17

    accuracy                         0.6725       171
   macro avg     0.6440    0.6597    0.6482       171
weighted avg     0.6626    0.6725    0.6640       171

